# Extract unified raster set for OCR + YOLO

This notebook builds **one folder of images** for downstream pipelines:

1. **All original photos** (jpg/png/…) are copied into `images/` (preserving format).
2. **Every video** contributes **one middle frame** saved as `.jpg`.

A **`unified_manifest.csv`** records every output file, original path, station id, and (for videos) the frame index used — so you can run **OCR** and **YOLO** in other notebooks without re-scanning `all_data/`.

## Output layout

```
HackMITChina2026/unified_media/
  images/                 # flat folder; unique filenames
  unified_manifest.csv    # index for OCR / YOLO / RN matrix
```

## Prerequisites

Run the EDA manifest cell first so `eda_outputs/media_manifest.csv` exists (or set `SOURCE_MANIFEST` below).

In [ ]:
%pip -q install opencv-python pandas tqdm

In [ ]:
from __future__ import annotations

import hashlib
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- paths (edit if needed) ---
REPO = Path("/home/chengwei/boartraining/HackMITChina2026")
SOURCE_MANIFEST = REPO / "eda_outputs" / "media_manifest.csv"
UNIFIED_ROOT = REPO / "unified_media"
IMAGES_DIR = UNIFIED_ROOT / "images"
MANIFEST_OUT = UNIFIED_ROOT / "unified_manifest.csv"


UNIFIED_ROOT.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print("SOURCE_MANIFEST:", SOURCE_MANIFEST)
print("UNIFIED_ROOT:", UNIFIED_ROOT)


In [ ]:
def path_fingerprint(p: Path) -> str:
    return hashlib.sha256(p.as_posix().encode("utf-8")).hexdigest()[:12]


def unique_dest_name(station_id: str, src: Path, suffix: str, media_kind: str) -> Path:
    """Flat unique filename under images/."""
    stem = src.stem[:120] if len(src.stem) > 120 else src.stem
    fp = path_fingerprint(src)
    return IMAGES_DIR / f"{station_id}__{media_kind}__{fp}__{stem}{suffix}"


def read_video_middle_frame_bgr(video_path: Path, max_dim: int = 1920) -> tuple[np.ndarray | None, int | None]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None, None
    fc = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    if fc is None or fc <= 0:
        cap.release()
        return None, None
    n = int(fc)
    mid = max(0, (n - 1) // 2)
    frame = None
    used = None
    for idx in (mid, max(0, mid - 1), min(n - 1, mid + 1)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, fr = cap.read()
        if ok and fr is not None:
            frame = fr
            used = int(idx)
            break
    cap.release()
    if frame is None:
        return None, None
    h, w = frame.shape[:2]
    scale = min(1.0, max_dim / max(h, w))
    if scale < 1.0:
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    return frame, used


def row_source_size_bytes(row: pd.Series, src: Path) -> int:
    if "size_bytes" not in row.index:
        return int(src.stat().st_size)
    v = row["size_bytes"]
    if pd.isna(v):
        return int(src.stat().st_size)
    return int(v)


def copy_image_row(row: pd.Series) -> dict:
    src = Path(row["path"])
    station_id = str(row["station_id"])
    station_raw = str(row["station_raw"])
    ext = src.suffix.lower()
    dst = unique_dest_name(station_id, src, ext, "img")
    shutil.copy2(src, dst)
    return {
        "unified_relpath": str(dst.relative_to(UNIFIED_ROOT)),
        "unified_path": str(dst.resolve()),
        "source_path": str(src),
        "source_media_type": "image",
        "station_id": station_id,
        "station_raw": station_raw,
        "original_filename": src.name,
        "source_suffix": ext,
        "video_middle_frame_index": np.nan,
        "source_size_bytes": row_source_size_bytes(row, src),
    }


def extract_video_middle_row(row: pd.Series) -> dict | None:
    src = Path(row["path"])
    station_id = str(row["station_id"])
    station_raw = str(row["station_raw"])
    frame, used_idx = read_video_middle_frame_bgr(src)
    if frame is None:
        return None
    dst = unique_dest_name(station_id, src, ".jpg", "vidframe")
    ok = cv2.imwrite(str(dst), frame)
    if not ok:
        return None
    return {
        "unified_relpath": str(dst.relative_to(UNIFIED_ROOT)),
        "unified_path": str(dst.resolve()),
        "source_path": str(src),
        "source_media_type": "video",
        "station_id": station_id,
        "station_raw": station_raw,
        "original_filename": src.name,
        "source_suffix": src.suffix.lower(),
        "video_middle_frame_index": float(used_idx) if used_idx is not None else np.nan,
        "source_size_bytes": row_source_size_bytes(row, src),
    }

print("Helpers ready.")


## Run extraction

- Set `LIMIT_ROWS` to an integer for a quick test, or `None` for full pass.
- Re-running overwrites files with the **same** destination name (same source path → same hash → same name). To force a full rebuild, delete `unified_media/images/` first.

In [ ]:
manifest = pd.read_csv(SOURCE_MANIFEST)
manifest["media_type"] = manifest["media_type"].astype(str)

LIMIT_ROWS = None  # e.g. 500 for smoke test
if LIMIT_ROWS is not None:
    manifest = manifest.head(LIMIT_ROWS).copy()

rows_out: list[dict] = []
errors: list[tuple[str, str]] = []

for _, row in tqdm(manifest.iterrows(), total=len(manifest)):
    mt = row["media_type"]
    src = Path(row["path"])
    if not src.is_file():
        errors.append((str(src), "missing_file"))
        continue
    try:
        if mt == "image":
            rows_out.append(copy_image_row(row))
        elif mt == "video":
            rec = extract_video_middle_row(row)
            if rec is None:
                errors.append((str(src), "video_frame_failed"))
            else:
                rows_out.append(rec)
        else:
            errors.append((str(src), f"unknown_media_type:{mt}"))
    except Exception as e:
        errors.append((str(src), repr(e)))

unified = pd.DataFrame(rows_out)
unified.to_csv(MANIFEST_OUT, index=False)

print("Wrote:", MANIFEST_OUT, "rows:", len(unified))
print("Errors:", len(errors))
if errors[:5]:
    print("Sample errors:", errors[:5])

## Next steps

### OCR (same or other notebook)
- Read `unified_manifest.csv`, and process images from `unified_path`.
- Write OCR output with key columns preserved: `unified_path`, `unified_relpath`, `source_path`, `station_id`.

### YOLO (separate notebook)
- Run inference on `unified_path` images.
- Save predictions with the same key columns: `unified_path`, `unified_relpath`, `source_path`, `station_id`.

### RN / matrix
- Recommended merge priority: `unified_path` (primary) -> `source_path` (fallback), then combine with timestamp and boar counts.